In [ ]:
# Következő meccsek adatai

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from nba_api.stats.endpoints import scoreboardv2
import nba_api_module as nbam
import time
import json

team_id_dict = nbam.TEAM_IDS
team_abr_dict = nbam.TEAM_ABBREVIATIONS

def get_upcoming_games(days_ahead=3):
    """
    Lekéri a következő N napra tervezett meccseket.
    Maximum 1 meccs csapatonként.
    """
    upcoming = []
    teams_seen = set()
    
    today = datetime.now()
    
    for day_offset in range(days_ahead):
        check_date = today + timedelta(days=day_offset)
        date_str = check_date.strftime('%Y-%m-%d')
        
        print(f"\nEllenőrzés: {date_str}")
        
        try:
            scoreboard = scoreboardv2.ScoreboardV2(game_date=date_str)
            games = scoreboard.get_data_frames()[0]  # GameHeader
            
            if len(games) == 0:
                print(f"  Nincs meccs ezen a napon")
                continue
            
            for _, game in games.iterrows():
                game_id = str(game['GAME_ID'])
                home_team_id = game['HOME_TEAM_ID']
                home_team = team_id_dict[home_team_id]
                home_team_abr = team_abr_dict[home_team]
                away_team_id = game['VISITOR_TEAM_ID']
                away_team = team_id_dict[away_team_id]
                away_team_abr = team_abr_dict[away_team]
                
                # Csak akkor adjuk hozzá, ha egyik csapat sem szerepelt még
                if home_team_id not in teams_seen and away_team_id not in teams_seen:
                    upcoming.append({
                        'game_id': game_id,
                        'game_date': date_str,
                        'home_team_id': home_team_id,
                        'away_team_id': away_team_id,
                        'home_team': home_team,
                        'away_team': away_team
                    })
                    teams_seen.add(home_team_id)
                    teams_seen.add(away_team_id)
                    print(f"  ✓ {home_team_abr} vs. {away_team_abr} (ID: {game_id})")
            
            time.sleep(1)  # Rate limit
            
        except Exception as e:
            print(f"  Hiba {date_str} lekérésekor: {e}")
            continue
    
    print(f"\n{'='*50}")
    print(f"Összesen {len(upcoming)} meccs találva")
    print(f"{'='*50}")
    
    return pd.DataFrame(upcoming)


def create_pregame_features(game_id, season, game_date, team_ids):
    """
    Egy adott meccsre létrehozza az összes pregame feature-t.
    """
    print(f"\n[{game_id}] Pregame features létrehozása...")
    
    features = {}
    
    # 1) Pregame stats
    try:
        pregame = nbam.extract_pregame(game_id, season, game_date, team_ids)
        features.update(pregame)
        print(f"  ✓ Pregame stats")
    except Exception as e:
        print(f"  ✗ Pregame stats hiba: {e}")
        return None
    
    time.sleep(1)
    
    # 2) Injury data
    try:
        injury = nbam.extract_injury(None)
        # Eltávolítjuk a missing_starters feature-t (mint az ml.ipynb-ben)
        injury.pop('home_missing_starters', None)
        injury.pop('away_missing_starters', None)
        features.update(injury)
        print(f"  ✓ Injury data")
    except Exception as e:
        print(f"  ✗ Injury data hiba: {e}")
        # Ha nincs injury adat, nullázzuk
        features['home_injury_count'] = 0
        features['away_injury_count'] = 0
    
    time.sleep(0.01)
    
    # 3) Advanced stats
    try:
        advanced = nbam.extract_advanced_stats(game_id, game_date, season, home_id=team_ids[0], away_id=team_ids[1])
        features.update(advanced)
        print(f"  ✓ Advanced stats")
    except Exception as e:
        print(f"  ✗ Advanced stats hiba: {e}")
        return None
    
    time.sleep(1)
    
    # 4) Form metrics
    try:
        form = nbam.extract_form(game_id, season, game_date, home_id=team_ids[0], away_id=team_ids[1])
        features.update(form)
        print(f"  ✓ Form metrics")
    except Exception as e:
        print(f"  ✗ Form metrics hiba: {e}")
        return None
    
    return features


def calculate_differential_features(features_dict):
    """
    Differenciális feature-ök hozzáadása (mint az ml.ipynb feature engineering cellájában)
    """
    df = pd.DataFrame([features_dict])
    
    # Offensive/Defensive Rating különbségek
    df['ORtg_diff'] = df['home_ORtg'] - df['away_ORtg']
    df['DRtg_diff'] = df['home_DRtg'] - df['away_DRtg']
    df['NET_rtg_diff'] = df['home_NET_rtg'] - df['away_NET_rtg']
    
    # Pace különbség
    df['PACE_diff'] = df['home_PACE'] - df['away_PACE']
    
    # Hatékonyság különbségek
    df['TS_diff'] = df['home_TS%'] - df['away_TS%']
    df['EFG_diff'] = df['home_EFG%'] - df['away_EFG%']
    df['AST_ratio_diff'] = df['home_AST_ratio'] - df['away_AST_ratio']
    df['OREB_diff'] = df['home_OREB%'] - df['away_OREB%']
    df['turnover_diff'] = df['home_turnover_ratio'] - df['away_turnover_ratio']
    
    # Játékos minőség különbségek
    df['starter_PER_diff'] = df['home_starter_avg_PER'] - df['away_starter_avg_PER']
    df['bench_PER_diff'] = df['home_bench_avg_PER'] - df['away_bench_avg_PER']
    df['star_usage_diff'] = df['home_star_usage'] - df['away_star_usage']
    df['avg_TS_diff'] = df['home_avg_TS'] - df['away_avg_TS']
    df['top3_points_diff'] = df['home_top3_points_avg'] - df['away_top3_points_avg']
    
    # Pihenés és forma különbségek
    df['rest_days_diff'] = df['home_rest_days'] - df['away_rest_days']
    df['recent_form10_diff'] = df['home_recent_form10'] - df['away_recent_form10']
    df['recent_form5_diff'] = df['home_recent_form5'] - df['away_recent_form5']
    df['recent_form3_diff'] = df['home_recent_form3'] - df['away_recent_form3']
    
    # Sérülés különbség
    df['injury_count_diff'] = df['home_injury_count'] - df['away_injury_count']
    
    # Back-to-back advantage
    df['b2b_advantage'] = df['away_is_back_to_back'].astype(int) - df['home_is_back_to_back'].astype(int)
    
    return df.iloc[0].to_dict()


def align_features_to_model(features_dict, feature_columns_path='models/feature_columns.json'):
    """
    Biztosítja, hogy a feature-ök pontosan egyezzenek a modell által elvárt feature listával.
    """
    with open(feature_columns_path, 'r') as f:
        expected_features = json.load(f)
    
    # Ellenőrizzük, hogy minden szükséges feature megvan-e
    missing_features = [f for f in expected_features if f not in features_dict]
    if missing_features:
        print(f"\n⚠️  Hiányzó features: {missing_features}")
        # Nullával töltjük fel a hiányzó feature-öket
        for feat in missing_features:
            features_dict[feat] = 0
    
    # Csak a modell által elvárt feature-öket tartjuk meg, megfelelő sorrendben
    aligned_features = {feat: features_dict[feat] for feat in expected_features}
    
    return pd.DataFrame([aligned_features])


def prepare_upcoming_games_for_prediction(days_ahead=3, output_file='data/upcoming_games_features.csv'):
    """
    Teljes pipeline: lekéri a következő meccseket és előkészíti a predikcióhoz.
    """
    print("="*60)
    print("KÖVETKEZŐ MECCSEK ELŐKÉSZÍTÉSE PREDIKCIÓHOZ")
    print("="*60)
    
    # 1) Következő meccsek lekérése
    upcoming_df = get_upcoming_games(days_ahead=days_ahead)
    
    if len(upcoming_df) == 0:
        print("\n❌ Nem találtunk következő meccseket!")
        return None
    
    # 2) Features gyűjtése minden meccshez
    all_features = []
    
    for idx, game in upcoming_df.iterrows():
        game_id = game['game_id']
        game_date = game['game_date']
        
        print(f"\n{'='*60}")
        print(f"[{idx+1}/{len(upcoming_df)}] {game['away_team']} @ {game['home_team']}")
        print(f"{'='*60}")
        
        try:
            # Pregame features
            features = create_pregame_features(game_id, season='2025-26', game_date=game_date, team_ids=[game['home_team_id'], game['away_team_id']])
            
            if features is None:
                print(f"  ⚠️  Kihagyva (hiányos adatok)")
                continue
            
            # Differenciális features
            features = calculate_differential_features(features)
            
            # Game meta info hozzáadása
            features['game_id'] = game_id
            features['game_date'] = game_date
            features['home_team'] = game['home_team']
            features['away_team'] = game['away_team']
            
            all_features.append(features)
            
            print(f"  ✅ Sikeres feldolgozás")
            
        except Exception as e:
            print(f"  ❌ Hiba: {e}")
            continue
        
        time.sleep(2)  # Rate limit
    
    if len(all_features) == 0:
        print("\n❌ Egyetlen meccshez sem sikerült adatot gyűjteni!")
        return None
    
    # 3) DataFrame összeállítása
    features_df = pd.DataFrame(all_features)
    
    # 4) Feature alignment a modell által elvárt formátumra
    print(f"\n{'='*60}")
    print("FEATURE ALIGNMENT")
    print(f"{'='*60}")
    
    # Meta információkat külön tároljuk
    meta_cols = ['game_id', 'game_date', 'home_team', 'away_team']
    meta_df = features_df[meta_cols].copy()
    
    # Feature-ök alignment
    aligned_features = []
    for idx, row in features_df.iterrows():
        row_dict = row.to_dict()
        aligned = align_features_to_model(row_dict)
        aligned_features.append(aligned)
    
    aligned_df = pd.concat(aligned_features, ignore_index=True)
    
    # Meta info visszacsatolása
    final_df = pd.concat([meta_df.reset_index(drop=True), aligned_df], axis=1)
    
    # 5) Mentés
    #final_df.to_csv(output_file, index=False)
    
    print(f"\n{'='*60}")
    print("✅ KÉSZ!")
    print(f"{'='*60}")
    print(f"Meccsek száma: {len(final_df)}")
    print(f"Feature-ök száma: {len(aligned_df.columns)}")
    print(f"Mentve: {output_file}")
    print(f"\nMeccsek:")
    for _, game in final_df.iterrows():
        print(f"  • {game['away_team']} @ {game['home_team']} ({game['game_date']})")
    
    return final_df


# HASZNÁLAT:
upcoming_features = prepare_upcoming_games_for_prediction(days_ahead=3)

upcoming_features

In [ ]:
# Predikciók készítése az összes modellel

import pandas as pd
import numpy as np
import joblib
import json

# -------------------------------------------------------------------------
# 1) ADATOK ÉS MODELLEK BETÖLTÉSE
# -------------------------------------------------------------------------

# Feature-ök betöltése
df = upcoming_features.copy()

# Meta oszlopok elkülönítése
meta_cols = ['game_id', 'game_date', 'home_team', 'away_team']
meta_df = df[meta_cols].copy()

# Feature columns betöltése
with open('models/feature_columns.json', 'r') as f:
    feature_cols = json.load(f)

X = df[feature_cols]

# Közös scaler betöltése
scaler = joblib.load('models/scaler.joblib')
X_scaled = scaler.transform(X)

print(f"Meccsek száma: {len(X)}")
print(f"Feature-ök száma: {len(feature_cols)}")

# -------------------------------------------------------------------------
# 2) PREDIKCIÓK - MINDEN MODELLRE
# -------------------------------------------------------------------------

predictions = meta_df.copy()

# --- Random Forest + PCA ---
pca_rf = joblib.load('models/rf_pca_transformer.joblib')
model_rf = joblib.load('models/rf_pca_model.joblib')

X_pca_rf = pca_rf.transform(X_scaled)
predictions['rf_pca_prob_home'] = model_rf.predict_proba(X_pca_rf)[:, 1]

print(f"\n✓ Random Forest + PCA ({pca_rf.n_components_} komponens)")

# --- XGBoost Very Shallow + PCA ---
pca_xgb_shallow = joblib.load('models/xgb_shallow_pca_transformer.joblib')
model_xgb_shallow = joblib.load('models/xgb_shallow_model.joblib')

X_pca_xgb_shallow = pca_xgb_shallow.transform(X_scaled)
predictions['xgb_shallow_prob_home'] = model_xgb_shallow.predict_proba(X_pca_xgb_shallow)[:, 1]

print(f"✓ XGBoost Shallow + PCA ({pca_xgb_shallow.n_components_} komponens)")

# --- XGBoost Tuned + PCA ---
pca_xgb_tuned = joblib.load('models/xgb_tuned_pca_transformer.joblib')
model_xgb_tuned = joblib.load('models/xgb_tuned_model.joblib')

X_pca_xgb_tuned = pca_xgb_tuned.transform(X_scaled)
predictions['xgb_tuned_prob_home'] = model_xgb_tuned.predict_proba(X_pca_xgb_tuned)[:, 1]

print(f"✓ XGBoost Tuned + PCA ({pca_xgb_tuned.n_components_} komponens)")

# -------------------------------------------------------------------------
# 3) ENSEMBLE ÉS ÖSSZEGZÉS
# -------------------------------------------------------------------------

# Ensemble átlag
predictions['ensemble_prob_home'] = predictions[['rf_pca_prob_home', 'xgb_shallow_prob_home', 'xgb_tuned_prob_home']].mean(axis=1)

# Away prob számítás
for col in ['rf_pca', 'xgb_shallow', 'xgb_tuned', 'ensemble']:
    predictions[f'{col}_prob_away'] = 1 - predictions[f'{col}_prob_home']

# -------------------------------------------------------------------------
# 4) EREDMÉNYEK MEGJELENÍTÉSE
# -------------------------------------------------------------------------

print(f"\n{'='*80}")
print("PREDIKCIÓK")
print(f"{'='*80}\n")

display_cols = ['home_team', 'away_team', 'rf_pca_prob_home', 'xgb_shallow_prob_home', 
                'xgb_tuned_prob_home', 'ensemble_prob_home']

print(predictions[display_cols].to_string(index=False))

# Mentés
#predictions.to_csv('data/predictions_upcoming.csv', index=False)
#print(f"\n✅ Mentve: data/predictions_upcoming.csv")

In [ ]:
# Oddsok

from pathlib import Path
import os
import sys

teams_tx_map = {
    "Atlanta Hawks":"Atlanta",
    "Miami Heat":"Miami",
    "Orlando Magic":"Orlando",
    "New York Knicks":"New York",
    "Milwaukee Bucks":"Milwaukee",
    "Phoenix Suns":"Phoenix",
    "Dallas Mavericks":"Dallas",
    "Minnesota Timberwolves":"Minnesota",
    "Toronto Raptors":"Toronto",
    "Cleveland Cavaliers":"Cleveland",
    "Brooklyn Nets":"Brooklyn",
    "Los Angeles Lakers":"LA Lakers",
    "Houston Rockets":"Houston",
    "New Orleans Pelicans":"New Orleans",
    "Golden State Warriors":"Golden State",
    "Boston Celtics":"Boston",
    "Denver Nuggets":"Denver",
    "San Antonio Spurs":"San Antonio",
    "LA Clippers":"LA Clippers",
    "Chicago Bulls":"Chicago",
    "Indiana Pacers":"Indiana",
    "Portland Trail Blazers":"Portland",
    "Philadelphia 76ers":"Philadelphia",
    "Oklahoma City Thunder":"Oklahoma City",
    "Utah Jazz":"Utah",
    "Sacramento Kings":"Sacramento",
    "Detroit Pistons":"Detroit",
    "Memphis Grizzlies":"Memphis",
    "Washington Wizards":"Washington"
}

teams_tx_map_rev = {v:k for k,v in teams_tx_map.items()}

nb_dir = Path(os.getcwd())
root = nb_dir.parents[2]
sys.path.append(str(root / "modules"))

from tx_module import get_league_odds

url = "https://www.tippmixpro.hu/hu/fogadas/i/bajnoksag-lokacio/kosarlabda/8/usa/229/nba/274663790763708416"
odds = get_league_odds(url)

odds['home_team_api'] = odds['home_team'].map(teams_tx_map_rev)
odds['away_team_api'] = odds['away_team'].map(teams_tx_map_rev)

preds_w_odds = pd.merge(predictions, 
                        odds[['home_team_api', 'away_team_api','home_odds', 'away_odds']],
                        left_on=['home_team', 'away_team'],
                        right_on=['home_team_api', 'away_team_api'],
                        how="inner")

preds_w_odds.drop(columns=['home_team_api', 'away_team_api'], inplace=True)
display(preds_w_odds)

In [ ]:
# Value betek

preds_w_odds['home_implied'] = 1/preds_w_odds['home_odds']
preds_w_odds['away_implied'] = 1/preds_w_odds['away_odds']

# find value 
for i, game in preds_w_odds.iterrows():
    if game['home_implied'] <= game['rf_pca_prob_home']:
        print(f"Bet on {game['home_team']} @ {game['home_odds']} (vs. {game['away_team']})")
    elif game['away_implied'] <= game['rf_pca_prob_away']:
        print(f"Bet on {game['away_team']} @ {game['away_odds']} (@ {game['home_team']})")


# add to existing csv
path = "data/value_bets_2025_26.csv"

prev_csv = pd.read_csv(path, dtype={"game_id": str})
new_csv = pd.concat([prev_csv, preds_w_odds], ignore_index=True).drop_duplicates(subset="game_id", keep="first")
new_csv.to_csv(path, index=False)
print(f"{len(new_csv) - len(prev_csv)} new rows added")

In [ ]:
# Paperbets

import nba_api_module as nbam
import pandas as pd

path_pbets = "data/paperbets.csv"

vbets = pd.read_csv("data/value_bets_2025_26.csv", dtype={"game_id": str})

bets_list = []
models = ['rf_pca', 'xgb_shallow', 'xgb_tuned', 'ensemble']
strats = ['fixed']
for i, game in vbets.iterrows():
    # Iterate for each model
    for model_name in models:
        # Find value
        prob_home_mod = game[f"{model_name}_prob_home"]
        prob_away_mod = game[f"{model_name}_prob_away"]
        prob_home_impl = game["home_implied"]
        prob_away_impl = game["away_implied"]
        odds_home = game['home_odds']
        odds_away = game['away_odds']

        if  prob_home_mod > prob_home_impl:
            bet, prob, odds = "home", prob_home_mod, odds_home
        elif prob_away_mod > prob_away_impl:
            bet, prob, odds = "away", prob_away_mod, odds_away
        else:
            bet, prob, odds = "-", 0, 0
        
        value = (odds*prob) - 1
        # Iterate for each strategy
        for strat in strats:
            if strat == "fixed":
                stake = 1 if value > 0 else 0
            else:
                stake = 0

            bet = {'game_id': game['game_id'],
                'model': model_name,
                'strategy': strat,
                'bet': bet,
                'prob': prob,
                'odds': odds,
                'value': value,
                'stake': stake,
                'WL': None,
                'profit': None
                }

            bets_list.append(bet)

pbets = pd.DataFrame(bets_list)

pbets_prev = pd.read_csv(path_pbets, dtype={'game_id': str})
count_none = len(pbets_prev[(pbets_prev['WL'].isna()) | (pbets_prev['WL'] == '-')])

# Evaluate paperbets
game_log = nbam.get_season_game_log("2025-26")
profit_new = 0
for i, bet in pbets_prev.iterrows():
    if pd.notna(bet['WL']):
        continue  # Already evaluated

    game_id = bet['game_id']
    home_mask = (game_log.MATCHUP.str.contains(" vs. "))
    game_row = game_log[(game_log.GAME_ID == game_id) & home_mask]
    if len(game_row) == 0:
        print(f"Game ID {game_id} not found in game log.")
        continue

    home_win = game_row.iloc[0]['PLUS_MINUS'] > 0
    if bet['bet'] == "home":
        wl = "W" if home_win else "L"
    elif bet['bet'] == "away":
        wl = "W" if not home_win else "L"
    else:
        wl = "-"
    
    # Calculate profit
    if bet['stake'] > 0:
        if wl == "W":
            profit = bet['odds'] * bet['stake'] - bet['stake']
        elif wl == "L":
            profit = -bet['stake']
        else:
            profit = 0
    else:
        profit = 0

    profit_new += profit if profit is not None else 0
    
    pbets_prev.at[i, 'WL'] = wl
    pbets_prev.at[i, 'profit'] = profit

count_none2 = len(pbets_prev[(pbets_prev['WL'].isna()) | (pbets_prev['WL'] == '-')])
print(f"Updated paperbets with {count_none - count_none2} results. (Profit: {profit_new:.1f})")

# Results so far
res_list = []
for model in models:
    for strat in strats:
        subset = pbets_prev[(pbets_prev['model'] == model) & (pbets_prev['strategy'] == strat)]
        total_bets = len(subset[subset['WL'].notna() & (subset['WL'] != '-')])
        total_profit = subset['profit'].sum()
        winrate = len(subset[subset['WL'] == 'W']) / total_bets * 100 if total_bets > 0 else 0
        roi = (total_profit / total_bets) * 100 if total_bets > 0 else 0
        
        # create dict
        result = {
            'model': model,
            'strategy': strat,
            'total_bets': total_bets,
            'total_profit': total_profit,
            'winrate_%': winrate,
            'ROI_%': roi
        }
        
        res_list.append(result)
        results = pd.DataFrame(res_list)

display(results)    

# Save
pbets_new = pd.concat([pbets_prev, pbets], ignore_index=True).drop_duplicates(subset=['game_id', 'model', 'strategy'], keep='first')
pbets_new.to_csv(path_pbets, index=False)
print(f"Total new paperbets: {len(pbets_new) - len(pbets_prev)}")